In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Load source tables
market_summary = spark.table(
    get_gold_table("marts", "mart_item_price_summary")
)

dim_execution = spark.table(
    get_table("intermediate", "dim_execution")
)

print(f"Market summary records: {market_summary.count():,}")
print(f"Execution records: {dim_execution.count():,}")

In [0]:
# Prepare execution metadata
execution_metadata = (
    dim_execution
    .select(
        "execution_id",
        F.col("started_at").alias("execution_started_at"),
        F.col("finished_at").alias("execution_finished_at"),
        "execution_duration_seconds",
        F.col("status").alias("execution_status"),
        F.col("response_count").alias("execution_response_count"),
        F.col("item_count").alias("execution_item_count"),
        F.col("total_orders").alias("execution_total_orders")
    )
)

In [0]:
# Consolidate Buy and Sell metrics into one row per item
rel_market_overview = (
    market_summary
    .groupBy(
        "execution_id",
        "item_slug",
        "item_name",
        "product_slug",
        "category"
    )
    .agg(
        F.max("observed_at").alias("observed_at"),

        # General Buy metrics
        F.max(F.when(F.col("order_type") == "buy", F.col("total_offers"))).alias("buy_total_offers"),
        F.max(F.when(F.col("order_type") == "buy", F.col("min_price"))).alias("buy_min_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("max_price"))).alias("buy_max_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("avg_price"))).alias("buy_avg_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("median_price"))).alias("buy_median_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("p25_price"))).alias("buy_p25_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("p75_price"))).alias("buy_p75_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("market_spread"))).alias("buy_market_spread"),

        # General Sell metrics
        F.max(F.when(F.col("order_type") == "sell", F.col("total_offers"))).alias("sell_total_offers"),
        F.max(F.when(F.col("order_type") == "sell", F.col("min_price"))).alias("sell_min_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("max_price"))).alias("sell_max_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("avg_price"))).alias("sell_avg_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("median_price"))).alias("sell_median_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("p25_price"))).alias("sell_p25_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("p75_price"))).alias("sell_p75_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("market_spread"))).alias("sell_market_spread"),

        # Buy metrics within IQR
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_offers"))).alias("buy_within_iqr_offers"),
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_min_price"))).alias("buy_within_iqr_min_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_max_price"))).alias("buy_within_iqr_max_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_avg_price"))).alias("buy_within_iqr_avg_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_median_price"))).alias("buy_within_iqr_median_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_p25_price"))).alias("buy_within_iqr_p25_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_p75_price"))).alias("buy_within_iqr_p75_price"),
        F.max(F.when(F.col("order_type") == "buy", F.col("within_iqr_spread"))).alias("buy_within_iqr_spread"),

        # Sell metrics within IQR
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_offers"))).alias("sell_within_iqr_offers"),
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_min_price"))).alias("sell_within_iqr_min_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_max_price"))).alias("sell_within_iqr_max_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_avg_price"))).alias("sell_within_iqr_avg_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_median_price"))).alias("sell_within_iqr_median_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_p25_price"))).alias("sell_within_iqr_p25_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_p75_price"))).alias("sell_within_iqr_p75_price"),
        F.max(F.when(F.col("order_type") == "sell", F.col("within_iqr_spread"))).alias("sell_within_iqr_spread"),

        # Market quality metrics
        F.sum("total_offers").alias("total_market_offers"),
        F.sum("within_iqr_offers").alias("total_within_iqr_offers"),
        F.sum("outlier_offers").alias("total_outliers"),
        F.sum("insufficient_sample_offers").alias("total_insufficient_sample_offers")
    )
)

In [0]:
# Validate execution metadata grain
duplicate_executions = (
    execution_metadata
    .groupBy("execution_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_executions > 0:
    raise ValueError(
        f"Execution metadata contains {duplicate_executions} duplicated execution IDs"
    )

# Add execution metadata
rel_market_overview = (
    rel_market_overview
    .join(
        execution_metadata,
        on="execution_id",
        how="left"
    )
)

print(f"Market overview records: {rel_market_overview.count():,}")

In [0]:
display(
    rel_market_overview
    .orderBy("item_slug")
)

In [0]:
# Validate market overview grain and metrics
display(
    rel_market_overview.agg(
        F.count("*").alias("total_records"),
        F.countDistinct("execution_id", "item_slug").alias("unique_execution_items"),
        F.countDistinct("execution_id").alias("unique_executions"),
        F.sum("total_market_offers").alias("total_market_offers"),
        F.sum("total_within_iqr_offers").alias("total_within_iqr_offers"),
        F.sum("total_outliers").alias("total_outliers"),
        F.sum("total_insufficient_sample_offers").alias("total_insufficient_sample_offers")
    )
)

# Check duplicated execution-item combinations
duplicates = (
    rel_market_overview
    .groupBy("execution_id", "item_slug")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicated execution-item combinations: {duplicates.count():,}")

# Check missing execution metadata
missing_metadata = (
    rel_market_overview
    .filter(
        F.col("execution_started_at").isNull()
        | F.col("execution_status").isNull()
    )
    .count()
)

print(f"Records with missing execution metadata: {missing_metadata:,}")

if duplicates.count() > 0:
    raise ValueError("Duplicate execution-item combinations detected")

if missing_metadata > 0:
    raise ValueError("Missing execution metadata detected")

In [0]:
# Define target table
TARGET_TABLE = get_gold_table(
    "reporting",
    "rel_market_overview"
)

# Materialize reporting table
(
    rel_market_overview
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Table successfully saved: {TARGET_TABLE}")

In [0]:
# Validate materialized table
saved_overview = spark.table(TARGET_TABLE)

print(f"Saved records: {saved_overview.count():,}")

display(
    saved_overview
    .orderBy("item_slug")
)